# TTS Dataset Measurement

Reads a manifest that already has audio and writes the same manifest back with
a `scores` object added to every row. It never moves or deletes audio, and it
never refuses a row: this notebook only measures. A later, separate notebook
reads the scores and decides what to keep.

The manifest format is [`docs/manifest.md`](../docs/manifest.md). This
notebook is step 4 (and step 7, for a converted manifest) of the chain
described in [`docs/pipeline.md`](../docs/pipeline.md). It never picks a donor
or a judge on its own — those come from [`docs/registry.md`](../docs/registry.md)
or from an explicit argument, per the pipeline's rules.

### Pipeline stages

| # | Stage | Description |
|---|---|---|
| 0 | Configuration | Set the manifest path and every parameter via env vars |
| 1 | Install dependencies | onnxruntime, speakeronnx, speechonnxmetrics, and (optionally) an OVOS STT plugin |
| 2 | Demo manifest | Only runs when `MANIFEST_PATH` does not exist yet: builds a small manifest from synthesised speech, so the notebook has something to measure on its own |
| 3 | Load manifests | Read the input manifest, an optional parent manifest, and any scores a previous run already wrote |
| 4 | Score every row | Naturalness, speaker similarity, prompt echo, length difference against the parent, and the STT round trip — each key computed only when its inputs exist |
| 5 | Write the scored manifest | `<name>.scored.jsonl` beside the input, plus its `.meta.json` sidecar, and the printed score distributions |

### Inputs
- `MANIFEST_PATH`: a JSON Lines manifest with at least `id`, `text`, `audio`,
  `duration_s` on every row (see `docs/manifest.md`). Audio paths in the
  manifest are relative to the manifest's own directory.
- Optionally `PARENT_MANIFEST`, when a row's `parent_id` points at a row that
  lives in a different manifest than the one being scored.
- Optionally one `REFERENCE_<IDENTITY>` WAV path per voice identity the
  manifest uses (identity upper-cased), for `spk_cos` and `echo_cos`.
- Optionally `STT_MODULE` (an OVOS STT plugin) and `STT_CONFIG`, for the
  recogniser round trip.

### Outputs
- `<name>.scored.jsonl`: the input manifest, unchanged except that every row
  now carries a `scores` object.
- `<name>.scored.jsonl.meta.json`: the instrument behind every score key
  (package, version, model) and the distribution of every score.

### Hardware
Every model here runs on onnxruntime, CPU by default (see `ONNX_RUNTIME` and
`ONNX_PROVIDERS` below). No stage needs a GPU. The naturalness and speaker
models are downloaded once, on first use, and add up to a few hundred
megabytes — this is why the notebook is not executed automatically by CI (see
the banner below) and should be run somewhere with a normal broadband link.


> ⚠ **NOT EXECUTED** — This notebook downloads a few hundred megabytes of
> model weights on first use (naturalness and speaker-embedding models).
> It is validated for parse correctness by CI but not executed automatically.
> Run it locally or on Kaggle/Colab; CPU is enough for every stage.


---
## NGI0 Grant Attribution

> Developed by TigreGotico for OpenVoiceOS, funded by the
> [NGI0 Commons Fund](https://nlnet.nl/project/OpenVoiceOS) / NLnet grant **101135429**.
> European Commission Next Generation Internet programme — DG CONNECT,
> grant agreement No 101135429.
> Additional funding: Swiss State Secretariat for Education, Research and Innovation (SERI).

**NGI work item:** WP3 — TTS dataset measurement (chain step 4/7)
**CI execution:** ❌ parse-only — see banner above

---


---
## 0 · Global Configuration

All parameters are set as environment variables via `os.environ.setdefault()`.
Override any of them from your shell, a `.env` file, or `%env` magic **before**
running the config cell.

| Key env vars | |
|---|---|
| `MANIFEST_PATH` | The manifest to score. Audio paths in it are relative to its own directory. |
| `PARENT_MANIFEST` | A second manifest holding parent rows this manifest's `parent_id` values point at, when they are not rows of this manifest itself. Empty = every parent is looked up in `MANIFEST_PATH`. |
| `REFERENCE_<IDENTITY>` | One WAV path per voice identity in the manifest (identity upper-cased, e.g. `REFERENCE_MIRO`, `REFERENCE_DII`). Not a fixed key: set one for every identity your manifest carries. An identity with none set gets naturalness scores only. |


In [ ]:
import os


def _env(key, default=""):
    return os.environ.get(key, default)


def _flag(key):
    return _env(key, "0").strip().lower() in ("1", "true", "yes")


# ──────────────────────────────────────────────
# INPUT / OUTPUT
# ──────────────────────────────────────────────
os.environ.setdefault("MANIFEST_PATH", "./tts_measure_demo/manifest.jsonl")
# Any manifest with audio (docs/manifest.md). If this path does not exist,
# Stage 2 below writes a small demo manifest here so the notebook can run
# standalone. Point this at a real manifest to score real data instead.
os.environ.setdefault("PARENT_MANIFEST", "")
# Optional second manifest for parent_id rows this manifest does not contain.

# ──────────────────────────────────────────────
# NATURALNESS  (speechonnxmetrics)
# ──────────────────────────────────────────────
os.environ.setdefault("QUALITY_FAMILIES", "utmos,dnsmos")  # comma list: utmos, dnsmos, nisqa

# ──────────────────────────────────────────────
# ROUND TRIP  (OVOS STT plugin; intended: ovos-stt-plugin-rover)
#   No default engine is picked here: an empty STT_MODULE skips the round
#   trip entirely, and the sidecar records that it was skipped.
# ──────────────────────────────────────────────
os.environ.setdefault("STT_MODULE", "")    # e.g. ovos-stt-plugin-rover; empty = round trip skipped
os.environ.setdefault("STT_CONFIG", "{}")  # JSON plugin config, e.g. {"backends": [...]} for rover
os.environ.setdefault("STT_LANG", "en")    # fallback language for rows whose own "lang" field is absent

# ──────────────────────────────────────────────
# ONNXRUNTIME
# ──────────────────────────────────────────────
os.environ.setdefault("ONNX_RUNTIME", "cpu")     # cpu | cuda | rocm — chooses what Stage 1 installs
os.environ.setdefault("ONNX_PROVIDERS", "auto")  # comma list, or auto: CUDA, ROCm, MIGraphX, CPU (CPU always last)

# ──────────────────────────────────────────────
# RESUMABILITY
# ──────────────────────────────────────────────
os.environ.setdefault("RESCORE", "0")  # 1 = recompute every score, even for rows that already carry one

print("Configuration loaded.")
print(f"  Manifest        : {_env('MANIFEST_PATH')}")
print(f"  Parent manifest : {_env('PARENT_MANIFEST') or '(same manifest)'}")
print(f"  Naturalness     : {_env('QUALITY_FAMILIES')}")
print(f"  Round trip      : {_env('STT_MODULE') or '(off — STT_MODULE empty)'}")
print(f"  onnxruntime     : {_env('ONNX_RUNTIME')} / providers={_env('ONNX_PROVIDERS')}")
print(f"  Rescore         : {_flag('RESCORE')}")


---
## 1 · Install dependencies

Installs onnxruntime for the chosen backend, the two in-house measurement
packages, and — only when `STT_MODULE` is set — `ovos-plugin-manager` and the
named STT plugin itself.

| Key env vars | |
|---|---|
| `ONNX_RUNTIME` | `cpu` installs `onnxruntime`; `cuda` installs `onnxruntime-gpu`; `rocm` prints AMD's install instructions (no ROCm/MIGraphX wheel is on PyPI). |
| `STT_MODULE` | Its PyPI package name is the module name itself, e.g. `ovos-stt-plugin-rover`. |


In [ ]:
onnx_runtime_choice = _env("ONNX_RUNTIME", "cpu").strip().lower()
if onnx_runtime_choice == "cuda":
    !pip install --quiet --break-system-packages --pre "onnxruntime-gpu>=1.30.0"
elif onnx_runtime_choice == "rocm":
    print(
        "No onnxruntime ROCm/MIGraphX wheel is on PyPI. Install AMD's build for "
        "your ROCm version from https://onnxruntime.ai/docs/execution-providers/ "
        "then re-run this cell's import check."
    )
else:
    !pip install --quiet --break-system-packages --pre "onnxruntime>=1.30.0"

!pip install --quiet --break-system-packages --pre "speakeronnx>=0.0.2a3" "speechonnxmetrics>=0.0.2a3" "librosa>=1.0.0" "soundfile>=0.14.0" "numpy>=2.5.3" "tqdm>=4.70.1"

stt_module = _env("STT_MODULE").strip()
if stt_module:
    # STT_MODULE names an OVOS STT plugin entrypoint (e.g. ovos-stt-plugin-rover);
    # its PyPI package name is the same string.
    !pip install --quiet --break-system-packages --pre "ovos-plugin-manager>=2.12.4a1"
    !pip install --quiet --break-system-packages --pre {stt_module}
else:
    print("STT_MODULE not set — the round trip (hyp, cer, read_wer) is skipped.")


---
## 2 · Demo manifest

Only runs when `MANIFEST_PATH` does not already exist. It synthesises five
short clips with `edge-tts` and writes a tiny manifest that exercises every
score this notebook can produce: two rows in the `demo` identity's own voice
(one of them a `parent_id` child, for `len_diff_s`), one row in a different
voice (a cross-voice pair, for `spk_cos`), and one row whose `text` field
does not match what the clip actually says (for the round-trip check, once
`STT_MODULE` is set).

Point `MANIFEST_PATH` at a real manifest to skip this stage and score real
data instead; nothing here runs in that case.

| Key env vars | |
|---|---|
| `MANIFEST_PATH` | When absent on disk, this stage creates it. |


In [ ]:
from pathlib import Path

if Path(_env("MANIFEST_PATH")).exists():
    print(f"{_env('MANIFEST_PATH')} already exists — using it as-is, demo generation skipped.")
else:
    !pip install --quiet --break-system-packages --pre "edge-tts>=7.2.8" "nest-asyncio>=1.6.0"

    import asyncio
    import json
    import subprocess

    import nest_asyncio
    import soundfile as sf

    nest_asyncio.apply()
    import edge_tts

    manifest_path = Path(_env("MANIFEST_PATH"))
    demo_dir = manifest_path.parent
    demo_dir.mkdir(parents=True, exist_ok=True)

    REF_VOICE = "en-GB-SoniaNeural"    # the "demo" identity's own voice
    CROSS_VOICE = "en-US-GuyNeural"    # a different speaker, for a cross-voice comparison
    SENTENCE_REF = "This is the reference voice for the demo identity."
    SENTENCE_SRC = "Hey Mycroft, what is the weather like today in Lisbon?"
    SENTENCE_WRONG = "The quick brown fox jumps over the lazy dog."  # planted mismatch: the clip does not say this

    async def _synth(text, voice, out_mp3):
        await edge_tts.Communicate(text, voice).save(str(out_mp3))

    def _make_clip(name, text, voice):
        mp3_path = demo_dir / f"{name}.mp3"
        wav_path = demo_dir / f"{name}.wav"
        asyncio.run(_synth(text, voice, mp3_path))
        subprocess.run(
            ["ffmpeg", "-y", "-i", str(mp3_path), "-ar", "16000", "-ac", "1", str(wav_path)],
            check=True, capture_output=True,
        )
        mp3_path.unlink()
        info = sf.info(str(wav_path))
        return wav_path.name, info.frames / info.samplerate, info.samplerate

    ref_name, _ref_dur, _ref_sr = _make_clip("reference_demo", SENTENCE_REF, REF_VOICE)
    os.environ["REFERENCE_DEMO"] = str(demo_dir / ref_name)

    same_name, same_dur, same_sr = _make_clip("demo-same", SENTENCE_SRC, REF_VOICE)
    cross_name, cross_dur, cross_sr = _make_clip("demo-cross", SENTENCE_SRC, CROSS_VOICE)
    child_name, child_dur, child_sr = _make_clip("demo-child", SENTENCE_SRC, REF_VOICE)
    wrong_name, wrong_dur, wrong_sr = _make_clip("demo-wrong-text", SENTENCE_SRC, REF_VOICE)

    demo_rows = [
        {"id": "demo-same", "text": SENTENCE_SRC, "lang": "en-GB", "audio": same_name,
         "duration_s": same_dur, "sample_rate": same_sr, "identity": "demo",
         "source": {"kind": "tts", "plugin": "edge-tts", "voice": REF_VOICE}},
        {"id": "demo-cross", "text": SENTENCE_SRC, "lang": "en-US", "audio": cross_name,
         "duration_s": cross_dur, "sample_rate": cross_sr, "identity": "demo",
         "source": {"kind": "tts", "plugin": "edge-tts", "voice": CROSS_VOICE}},
        {"id": "demo-child", "parent_id": "demo-same", "text": SENTENCE_SRC, "lang": "en-GB",
         "audio": child_name, "duration_s": child_dur, "sample_rate": child_sr, "identity": "demo",
         "source": {"kind": "tts", "plugin": "edge-tts", "voice": REF_VOICE}},
        {"id": "demo-wrong-text", "text": SENTENCE_WRONG, "lang": "en-GB", "audio": wrong_name,
         "duration_s": wrong_dur, "sample_rate": wrong_sr, "identity": "demo",
         "source": {"kind": "tts", "plugin": "edge-tts", "voice": REF_VOICE}},
    ]
    with open(manifest_path, "w", encoding="utf-8") as f:
        for row in demo_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"Wrote a {len(demo_rows)}-row demo manifest to {manifest_path}")
    print(f"Reference voice for the 'demo' identity: {os.environ['REFERENCE_DEMO']}")


---
## 3 · Load manifests

Reads `MANIFEST_PATH` and, when set, `PARENT_MANIFEST`. When a previous run
already wrote `<name>.scored.jsonl` for this manifest, its scores are carried
onto the rows loaded here before scoring starts, so pointing this notebook at
either the original manifest or its own previous output resumes the same way.

The output path is always `<name>.scored.jsonl` next to the input, unless the
input is itself already such a file (a rerun in place), in which case the
output overwrites it.


In [ ]:
import json
from pathlib import Path


def _read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def _output_manifest_path(manifest_path):
    p = Path(manifest_path)
    if p.name.endswith(".scored.jsonl"):
        return p
    if p.suffix == ".jsonl":
        return p.with_name(p.stem + ".scored.jsonl")
    return p.with_name(p.name + ".scored.jsonl")


def _sidecar_path(manifest_like_path):
    return Path(str(manifest_like_path) + ".meta.json")


manifest_path = Path(_env("MANIFEST_PATH"))
manifest_dir = str(manifest_path.resolve().parent)

rows = _read_jsonl(manifest_path)
rows_by_id = {r["id"]: r for r in rows if "id" in r}

parent_manifest = _env("PARENT_MANIFEST").strip()
parent_rows_by_id = {}
if parent_manifest:
    parent_rows_by_id = {r["id"]: r for r in _read_jsonl(parent_manifest) if "id" in r}

output_path = _output_manifest_path(manifest_path)
sidecar_path = _sidecar_path(output_path)

if output_path.resolve() != manifest_path.resolve() and output_path.exists():
    previous_by_id = {r["id"]: r for r in _read_jsonl(output_path) if "id" in r}
    for row in rows:
        prev = previous_by_id.get(row.get("id"))
        if prev and prev.get("scores"):
            row["scores"] = prev["scores"]
    print(f"Carried scores forward from a previous run at {output_path}")

print(f"Loaded {len(rows)} row(s) from {manifest_path}")
print(f"Parent lookups: {len(rows_by_id)} in this manifest" + (
    f", {len(parent_rows_by_id)} in {parent_manifest}" if parent_manifest else ""
))
print(f"Output will be written to {output_path}")


---
## 4 · Score every row

Each key is computed only when its inputs exist; otherwise it stays absent
from `scores` rather than being written as zero or null.

1. **Naturalness** — the `speechonnxmetrics` families named in
   `QUALITY_FAMILIES` (`utmos`, `dnsmos`, `nisqa` by default `utmos,dnsmos`),
   mapped onto the manifest's own key names (`utmos`, `dnsmos_sig`,
   `dnsmos_bak`, `dnsmos_ovrl`, `nisqa_mos`). Needs only the clip's audio.
2. **`spk_cos`** — cosine, from `speakeronnx`'s WeSpeaker ResNet34 embedder,
   between the clip and the row's identity reference (`REFERENCE_<IDENTITY>`).
   Only for rows that carry an `identity`, and only when that identity's
   reference variable is set and points at a file that exists.
3. **`echo_cos`** — cosine between a 200-bin mean-absolute-amplitude
   envelope of the clip and of the same identity reference, each
   L2-normalised. A high value means the clip is close to being the
   reference itself. Same precondition as `spk_cos`.
4. **`len_diff_s`** — this row's `duration_s` minus its parent's, for rows
   with a `parent_id` whose parent (in this manifest or in
   `PARENT_MANIFEST`) carries a `duration_s`. Needs no audio file, only the
   durations already on record.
5. **The round trip** — `hyp` is this clip transcribed by `STT_MODULE`
   (empty means every row skips this rule, and the sidecar records that the
   round trip was skipped). `cer` is the character error rate of `hyp`
   against the row's `text`, both normalised first (Unicode NFC, casefold,
   punctuation stripped, whitespace collapsed). `read_wer` is the word
   error rate between this row's `hyp` and its parent's `hyp` (same
   normalisation), when the parent has one.

A clip whose naturalness scoring fails, or whose STT call raises, prints a
warning and leaves that row's affected keys absent — one row's failure never
stops the rest of the manifest.


In [ ]:
import re
import sys
import unicodedata

import librosa
import numpy as np
import onnxruntime as ort
import soundfile as sf
from tqdm import tqdm

from speakeronnx import SpeakerEmbedder, cosine
from speechonnxmetrics import score as som_score
from speechonnxmetrics.asr import cer as som_cer


# ═══════════════════════════════════════════════════════════════════
# Text and audio helpers
# ═══════════════════════════════════════════════════════════════════

def _word_error_rate(ref_words: list, hyp_words: list) -> float:
    """Word-level Levenshtein distance divided by the reference length."""
    if not ref_words:
        return 0.0 if not hyp_words else 1.0
    prev = list(range(len(hyp_words) + 1))
    for i, r in enumerate(ref_words, 1):
        cur = [i] + [0] * len(hyp_words)
        for j, h in enumerate(hyp_words, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (r != h))
        prev = cur
    return prev[-1] / len(ref_words)


def _normalize_for_cer(text: str) -> str:
    """Unicode NFC, casefold, strip punctuation, collapse spaces."""
    text = unicodedata.normalize("NFC", text).casefold()
    text = "".join(" " if unicodedata.category(ch).startswith("P") else ch for ch in text)
    return re.sub(r"\s+", " ", text).strip()


def _envelope(wav: np.ndarray, n_bins: int = 200) -> np.ndarray:
    """Mean-absolute-amplitude envelope in n_bins equal bins, L2-normalised."""
    if len(wav) == 0:
        return np.zeros(n_bins, dtype=np.float32)
    edges = np.linspace(0, len(wav), n_bins + 1).astype(int)
    env = np.array([
        np.mean(np.abs(wav[edges[i]:edges[i + 1]])) if edges[i + 1] > edges[i] else 0.0
        for i in range(n_bins)
    ], dtype=np.float32)
    norm = np.linalg.norm(env)
    return env / norm if norm > 0 else env


def _make_stt(module_name: str, plugin_config: dict, lang: str):
    from ovos_plugin_manager.stt import load_stt_plugin
    cfg = dict(plugin_config)
    cfg.setdefault("lang", lang)
    cls = load_stt_plugin(module_name)
    return cls(config=cfg)


def _transcribe(stt, path: str, lang: str) -> str:
    from ovos_plugin_manager.utils.audio import AudioData
    audio = AudioData.from_file(path)
    return stt.execute(audio, language=lang)


PROVIDER_PRIORITY = [
    "CUDAExecutionProvider", "ROCMExecutionProvider", "MIGraphXExecutionProvider", "CPUExecutionProvider",
]


def _onnx_providers():
    raw = _env("ONNX_PROVIDERS", "auto").strip()
    if raw.lower() == "auto":
        available = set(ort.get_available_providers())
        ordered = [p for p in PROVIDER_PRIORITY if p in available]
        if "CPUExecutionProvider" not in ordered:
            ordered.append("CPUExecutionProvider")
        return ordered
    return [p.strip() for p in raw.split(",") if p.strip()]


# speechonnxmetrics keys -> docs/manifest.md keys
NATURALNESS_KEYS = {
    "utmos": "utmos",
    "dnsmos.sig": "dnsmos_sig",
    "dnsmos.bak": "dnsmos_bak",
    "dnsmos.ovrl": "dnsmos_ovrl",
    "nisqa.mos": "nisqa_mos",
}


def _resolve_audio(manifest_dir, row):
    audio = row.get("audio")
    if not audio:
        return None
    path = os.path.join(manifest_dir, audio)
    return path if os.path.exists(path) else None


# The speaker embedder and each identity's reference are loaded lazily: a
# manifest with no "identity" field never needs either.
_embedder_holder = {}


def _get_embedder():
    if "embedder" not in _embedder_holder:
        print("Loading speaker embedder (wespeaker-resnet34)...")
        _embedder_holder["embedder"] = SpeakerEmbedder(model="wespeaker-resnet34", providers=_onnx_providers())
    return _embedder_holder["embedder"]


def _get_reference(identity, ref_cache):
    if identity in ref_cache:
        return ref_cache[identity]
    ref_path = _env(f"REFERENCE_{identity.upper()}").strip()
    if not ref_path or not os.path.exists(ref_path):
        ref_cache[identity] = None
        return None
    wav, _sr = librosa.load(ref_path, sr=None, mono=True)
    ref_cache[identity] = {
        "path": ref_path,
        "envelope": _envelope(wav),
        "embedding": _get_embedder().embed(ref_path),
    }
    return ref_cache[identity]


# ═══════════════════════════════════════════════════════════════════
# Per-row scoring
# ═══════════════════════════════════════════════════════════════════

def score_row(row, manifest_dir, rows_by_id, parent_rows_by_id, families, providers, stt, stt_lang, ref_cache):
    scores = {}
    audio_path = _resolve_audio(manifest_dir, row)

    if audio_path:
        try:
            quality = som_score(audio_path, metrics=families, providers=providers)
        except Exception as exc:
            quality = {}
            print(f"  [{row.get('id')}] naturalness scoring failed: {exc}")
        for src_key, out_key in NATURALNESS_KEYS.items():
            value = quality.get(src_key)
            if value is not None:
                scores[out_key] = float(value)

        identity = row.get("identity")
        if identity:
            ref = _get_reference(identity, ref_cache)
            if ref is not None:
                clip_emb = _get_embedder().embed(audio_path)
                scores["spk_cos"] = float(cosine(clip_emb, ref["embedding"]))
                wav, _sr = librosa.load(audio_path, sr=None, mono=True)
                scores["echo_cos"] = float(np.dot(_envelope(wav), ref["envelope"]))

        if stt is not None:
            lang = row.get("lang") or stt_lang
            try:
                hyp = _transcribe(stt, audio_path, lang)
            except Exception as exc:
                hyp = None
                print(f"  [{row.get('id')}] STT failed: {exc}")
            if hyp is not None:
                scores["hyp"] = hyp
                text = row.get("text")
                if text:
                    try:
                        scores["cer"] = float(som_cer(_normalize_for_cer(text), _normalize_for_cer(hyp)))
                    except Exception as exc:
                        print(f"  [{row.get('id')}] CER failed: {exc}")
                parent_id = row.get("parent_id")
                if parent_id:
                    parent = rows_by_id.get(parent_id) or parent_rows_by_id.get(parent_id)
                    parent_hyp = (parent.get("scores") or {}).get("hyp") if parent else None
                    if parent_hyp:
                        ref_words = _normalize_for_cer(parent_hyp).split()
                        hyp_words = _normalize_for_cer(hyp).split()
                        scores["read_wer"] = float(_word_error_rate(ref_words, hyp_words))

    parent_id = row.get("parent_id")
    if parent_id and row.get("duration_s") is not None:
        parent = rows_by_id.get(parent_id) or parent_rows_by_id.get(parent_id)
        if parent is not None and parent.get("duration_s") is not None:
            scores["len_diff_s"] = float(row["duration_s"] - parent["duration_s"])

    return scores


# ═══════════════════════════════════════════════════════════════════
# Run: score every row that is not already scored (unless RESCORE=1)
# ═══════════════════════════════════════════════════════════════════

families = [f.strip() for f in _env("QUALITY_FAMILIES", "utmos,dnsmos").split(",") if f.strip()]
providers = _onnx_providers()
print(f"onnxruntime providers: {providers}")

rescore = _flag("RESCORE")
stt_module = _env("STT_MODULE").strip()
stt = None
if stt_module:
    stt_config = json.loads(_env("STT_CONFIG", "{}") or "{}")
    print(f"Loading STT plugin for the round trip: {stt_module}")
    stt = _make_stt(stt_module, stt_config, _env("STT_LANG", "en"))
else:
    print("STT_MODULE not set — the round trip (hyp, cer, read_wer) is skipped.")

ref_cache = {}
n_scored, n_skipped = 0, 0
for row in tqdm(rows, desc="Measure", unit="row", file=sys.stdout):
    if row.get("scores") and not rescore:
        n_skipped += 1
        continue
    row["scores"] = score_row(
        row, manifest_dir, rows_by_id, parent_rows_by_id,
        families, providers, stt, _env("STT_LANG", "en"), ref_cache,
    )
    n_scored += 1

print(f"\nScored {n_scored} row(s), skipped {n_skipped} already-scored row(s) (RESCORE={rescore}).")


---
## 5 · Write the scored manifest

Writes every row (scored this run or carried over) to `<name>.scored.jsonl`,
prints the distribution (min, p5, p50, p95, max) of every score, and how many
`echo_cos` values land within 0.05 of 0.85 — the band a clip that is close to
being the reference itself falls into; deciding a refusal band from it is the
select notebook's job, not this one's. The `.meta.json` sidecar records the
package, version and model behind every score key, or that the round trip was
skipped.


In [ ]:
import importlib.metadata


def _pkg_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def _write_jsonl(path, rows_to_write):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows_to_write:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


_write_jsonl(output_path, rows)

score_keys = sorted({
    k for r in rows for k, v in (r.get("scores") or {}).items() if isinstance(v, (int, float))
})

print("Score distributions:")
distributions = {}
for key in score_keys:
    values = [r["scores"][key] for r in rows if isinstance((r.get("scores") or {}).get(key), (int, float))]
    arr = np.array(values, dtype=float)
    p5, p50, p95 = np.percentile(arr, [5, 50, 95])
    distributions[key] = {
        "n": len(values), "min": float(arr.min()), "p5": float(p5),
        "p50": float(p50), "p95": float(p95), "max": float(arr.max()),
    }
    print(f"  {key:12s} n={len(values):4d}  min={arr.min():.4f}  p5={p5:.4f}  p50={p50:.4f}  p95={p95:.4f}  max={arr.max():.4f}")

ECHO_REPORT_CENTER, ECHO_REPORT_BAND = 0.85, 0.05
echo_values = [r["scores"]["echo_cos"] for r in rows if isinstance((r.get("scores") or {}).get("echo_cos"), (int, float))]
if echo_values:
    near = sum(1 for v in echo_values if abs(v - ECHO_REPORT_CENTER) <= ECHO_REPORT_BAND)
    print(f"  echo_cos: {near} of {len(echo_values)} clip(s) within {ECHO_REPORT_BAND} of {ECHO_REPORT_CENTER}")

instruments = {}
for src_key, out_key in NATURALNESS_KEYS.items():
    if out_key in score_keys:
        instruments[out_key] = {
            "package": "speechonnxmetrics", "version": _pkg_version("speechonnxmetrics"),
            "model": src_key.split(".")[0],
        }
if "spk_cos" in score_keys:
    instruments["spk_cos"] = {
        "package": "speakeronnx", "version": _pkg_version("speakeronnx"), "model": "wespeaker-resnet34",
    }
if "echo_cos" in score_keys:
    instruments["echo_cos"] = {
        "package": "tts_measure.ipynb", "version": None,
        "model": "200-bin mean-absolute-amplitude envelope, L2-normalised, cosine against the identity reference",
    }
if "len_diff_s" in score_keys:
    instruments["len_diff_s"] = {
        "package": "tts_measure.ipynb", "version": None, "model": "duration_s minus the parent row's duration_s",
    }
if stt_module:
    instruments["hyp"] = {"package": stt_module, "version": _pkg_version(stt_module), "model": stt_config}
    if "cer" in score_keys:
        instruments["cer"] = {
            "package": "speechonnxmetrics", "version": _pkg_version("speechonnxmetrics"),
            "model": "character error rate, NFC/casefold/punctuation-stripped/space-collapsed",
        }
    if "read_wer" in score_keys:
        instruments["read_wer"] = {
            "package": "tts_measure.ipynb", "version": None,
            "model": "word-level Levenshtein against the parent row's hyp, same normalisation as cer",
        }
else:
    instruments["round_trip"] = "skipped (STT_MODULE empty)"

sidecar = {
    "notebook": "tts_measure.ipynb",
    "manifest": str(manifest_path),
    "output": str(output_path),
    "rows_total": len(rows),
    "rows_scored_this_run": n_scored,
    "rows_already_scored": n_skipped,
    "instruments": instruments,
    "distributions": distributions,
}
with open(sidecar_path, "w", encoding="utf-8") as f:
    json.dump(sidecar, f, ensure_ascii=False, indent=2)

print(f"\nWrote {len(rows)} row(s) to {output_path}")
print(f"Sidecar: {sidecar_path}")
